# KATM API Walkthrough

This notebook provides a detailed walkthrough of all major KATM classes and their options.

## Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

In [ ]:
# Import all KATM components
from katm import (
    KATM,
    KATMFast,
    DocumentBuilder,
    KeyphraseExtractor,
    SentenceEmbedder,
    GMMTopicClusterer,
    WordTopicProjector,
    WordTopicProjectorFast,
    clean_text,
    build_vocabulary,
    extract_content_words
)

## DocumentBuilder

The `DocumentBuilder` class provides different strategies for building documents from raw texts.

In [ ]:
# Sample raw texts
sample_texts = [
    "This is the first paragraph. It contains some information about machine learning.\n\nThis is the second paragraph.",
    "Another document here. Python programming is great. Data science uses statistics."
]

In [ ]:
# Strategy: paragraph_group - splits on double newlines, groups into chunks
builder = DocumentBuilder(strategy='paragraph_group', chunk_size=5)
docs = builder.build(sample_texts)
print("paragraph_group strategy:")
for i, doc in enumerate(docs):
    print(f"  Doc {i}: {doc[:80]}...")

In [ ]:
# Strategy: fixed_window - splits into sentences, groups into chunks of N sentences
builder = DocumentBuilder(strategy='fixed_window', chunk_size=2)
docs = builder.build(sample_texts)
print("fixed_window strategy:")
for i, doc in enumerate(docs):
    print(f"  Doc {i}: {doc[:80]}...")

In [ ]:
# Strategy: post_group - groups chunk_size posts into one document
builder = DocumentBuilder(strategy='post_group', chunk_size=2)
docs = builder.build(sample_texts)
print("post_group strategy:")
for i, doc in enumerate(docs):
    print(f"  Doc {i}: {doc[:80]}...")

## KeyphraseExtractor

Standalone keyphrase extraction with multiple algorithms.

In [ ]:
sample_docs = [
    "Machine learning is a subset of artificial intelligence. Deep learning uses neural networks.",
    "Python is a popular programming language. Python is used in data science and web development."
]

In [ ]:
# KeyBERT extractor
extractor = KeyphraseExtractor(algorithm='keybert', n_keyphrases=5)
keyphrases = extractor.extract(sample_docs)
print("KeyBERT keyphrases:")
for i, kps in enumerate(keyphrases):
    print(f"  Doc {i}: {kps}")

In [ ]:
# RAKE extractor
extractor = KeyphraseExtractor(algorithm='rake', n_keyphrases=5)
keyphrases = extractor.extract(sample_docs)
print("RAKE keyphrases:")
for i, kps in enumerate(keyphrases):
    print(f"  Doc {i}: {kps}")

In [ ]:
# YAKE extractor
extractor = KeyphraseExtractor(algorithm='yake', n_keyphrases=5)
keyphrases = extractor.extract(sample_docs)
print("YAKE keyphrases:")
for i, kps in enumerate(keyphrases):
    print(f"  Doc {i}: {kps}")

In [ ]:
# TF-IDF extractor
extractor = KeyphraseExtractor(algorithm='tfidf', n_keyphrases=5)
keyphrases = extractor.extract(sample_docs)
print("TF-IDF keyphrases:")
for i, kps in enumerate(keyphrases):
    print(f"  Doc {i}: {kps}")

In [ ]:
# GSC (Greedy Semantic Coverage) extractor
extractor = KeyphraseExtractor(algorithm='gsc', n_keyphrases=5)
keyphrases = extractor.extract(sample_docs)
print("GSC keyphrases:")
for i, kps in enumerate(keyphrases):
    print(f"  Doc {i}: {kps}")

## SentenceEmbedder

Standalone sentence embedding using sentence-transformers.

In [ ]:
embedder = SentenceEmbedder(model_name='all-MiniLM-L6-v2')
texts = ["Machine learning is great", "Deep learning uses neural networks"]
embeddings = embedder.encode(texts)
print(f"Embedding shape: {embeddings.shape}")

## GMMTopicClusterer

Standalone GMM clustering for topic discovery.

In [ ]:
import numpy as np

# Generate random embeddings for demonstration
np.random.seed(42)
random_embeddings = np.random.randn(100, 384).astype(np.float32)

# Fit GMM clusterer
clusterer = GMMTopicClusterer(n_topics=4, random_state=42)
clusterer.fit(random_embeddings)

# Get cluster centers (topic centroids)
centroids = clusterer.cluster_centers_
print(f"Number of topics: {clusterer.n_topics}")
print(f"Centroid shape: {centroids.shape}")

In [ ]:
# Predict topic probabilities for new embeddings
new_embeddings = np.random.randn(5, 384).astype(np.float32)
probs = clusterer.predict_proba(new_embeddings)
print(f"Probability shape: {probs.shape}")

## WordTopicProjector

Projects vocabulary words onto topics via cosine similarity to GMM centroids.

In [ ]:
# Create a small vocabulary
vocab = [
    "machine", "learning", "data", "science", "neural", "network",
    "python", "programming", "algorithm", "model", "training",
    "deep", "artificial", "intelligence", "feature", "prediction"
]

# Create embedder and clusterer
embedder = SentenceEmbedder(model_name='all-MiniLM-L6-v2')
clusterer = GMMTopicClusterer(n_topics=3, random_state=42)

# For demo, fit clusterer on vocabulary embeddings
vocab_embeddings = embedder.encode(vocab)
clusterer.fit(vocab_embeddings)

# Project vocabulary onto topics
projector = WordTopicProjector(clusterer=clusterer, embedder=embedder)
topic_words = projector.project_vocabulary(vocab, min_prob=0.1)

for topic_id, words in topic_words.items():
    print(f"Topic {topic_id}: {words}")

## WordTopicProjectorFast

Faster version of WordTopicProjector using incremental MMR (S4 optimization).

In [ ]:
# Use the same setup with WordTopicProjectorFast
projector_fast = WordTopicProjectorFast(clusterer=clusterer, embedder=embedder)
topic_words_fast = projector_fast.project_vocabulary(vocab, min_prob=0.1)

for topic_id, words in topic_words_fast.items():
    print(f"Topic {topic_id}: {words}")

## Utility Functions

KATM provides utility functions for text processing.

In [ ]:
# clean_text: normalize and clean text
raw_text = "  This   is  a   test\u00a0with\x00nonprintable\u2028chars  "
cleaned = clean_text(raw_text)
print(f"Cleaned text: '{cleaned}'")

In [ ]:
# build_vocabulary: extract words appearing at least min_df times
texts = ["machine learning is great", "machine learning uses data", "data science is great"]
vocab = build_vocabulary(texts, min_df=2)
print(f"Vocabulary: {vocab}")

In [ ]:
# extract_content_words: extract content words using TF-IDF or spaCy
content_words = extract_content_words(texts, min_df=2, method='tfidf')
print(f"Content words (TF-IDF): {content_words}")